In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import resample


from sklearn.metrics import classification_report
from sklearn.metrics import average_precision_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## 1. 데이터 로드 및 기본 탐생

- 데이터 구조 확인
- 정상 거래와 사기 거래 건수 확인

In [3]:
# 데이터 로드, column 형태 확인

df = pd.read_csv('/content/creditcard.csv')
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


- 전체 Column은 PCA로 압축된 데이터 column으로 추정됨
  - 어떤 의미를 가지고 있는지 파악 불가

In [4]:
# 전체 데이터 구조 확인

df.describe()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.000000,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,...,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000
mean,94813.859575,1.168375e-15,3.416908e-16,-1.379537e-15,2.074095e-15,9.604066e-16,1.487313e-15,-5.556467e-16,1.213481e-16,-2.406331e-15,...,1.654067e-16,-3.568593e-16,2.578648e-16,4.473266e-15,5.340915e-16,1.683437e-15,-3.660091e-16,-1.227390e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,...,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,...,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
25%,54201.500000,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,...,-2.283949e-01,-5.423504e-01,-1.618463e-01,-3.545861e-01,-3.171451e-01,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000
50%,84692.000000,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,...,-2.945017e-02,6.781943e-03,-1.119293e-02,4.097606e-02,1.659350e-02,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000
75%,139320.500000,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,...,1.863772e-01,5.285536e-01,1.476421e-01,4.395266e-01,3.507156e-01,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,...,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [6]:
# 클래스 비율 확인

df['Class'].value_counts(normalize=True)

,proportion
Class,
0,0.998273
1,0.001727


## 2. 샘플링

- 사기 거래(class = 1) 유지
- 정상 거래(class = 0) 10,000건 무작위 샘플링
  - sampling시 random_state = 42

In [7]:
# 사기거래 유지, 정상거래 10,000건 무작위 샘플링

df_fraud = df[df['Class'] == 1]
df_normal =  resample(df[df['Class'] == 0], n_samples=10000, replace=False, random_state=42)

In [8]:
# 두 데이터셋 합쳐 새로운 분석용 데이터프레임 제작

df_sample = pd.concat([df_fraud, df_normal])
df_sample


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
541,406.0,-2.312227,1.951992,-1.609851,3.997906,-0.522188,-1.426545,-2.537387,1.391657,-2.770089,...,0.517232,-0.035049,-0.465211,0.320198,0.044519,0.177840,0.261145,-0.143276,0.00,1
623,472.0,-3.043541,-3.157307,1.088463,2.288644,1.359805,-1.064823,0.325574,-0.067794,-0.270953,...,0.661696,0.435477,1.375966,-0.293803,0.279798,-0.145362,-0.252773,0.035764,529.00,1
4920,4462.0,-2.303350,1.759247,-0.359745,2.330243,-0.821628,-0.075788,0.562320,-0.399147,-0.238253,...,-0.294166,-0.932391,0.172726,-0.087330,-0.156114,-0.542628,0.039566,-0.153029,239.93,1
6108,6986.0,-4.397974,1.358367,-2.592844,2.679787,-1.128131,-1.706536,-3.496197,-0.248778,-0.247768,...,0.573574,0.176968,-0.436207,-0.053502,0.252405,-0.657488,-0.827136,0.849573,59.00,1
6329,7519.0,1.234235,3.019740,-4.304597,4.732795,3.624201,-1.357746,1.713445,-0.496358,-1.282858,...,-0.379068,-0.704181,-0.656805,-1.632653,1.488901,0.566797,-0.010016,0.146793,1.00,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218680,141415.0,-0.762961,1.897243,1.931378,4.191413,0.103570,1.367957,-0.210296,0.799408,-1.949474,...,-0.217645,-0.639138,-0.096265,0.417441,0.001403,0.194527,0.236362,0.106503,0.76,0
239359,150069.0,-0.299711,1.079933,-0.500521,-0.571127,1.362166,-0.241336,1.061852,-0.055889,0.025168,...,-0.008621,0.287652,-0.302456,-0.025240,-0.037041,0.588618,0.369017,0.266397,25.99,0
262759,160634.0,2.129101,-0.873931,-1.635981,-1.176035,-0.073736,-0.412121,-0.289237,-0.223462,-0.776604,...,-0.034599,-0.262403,0.091163,-1.095939,-0.098260,-0.387646,-0.046397,-0.065703,73.04,0
62511,50297.0,1.127518,0.118124,0.339852,0.599886,-0.359735,-0.421149,-0.161974,0.141529,-0.043080,...,-0.201777,-0.683207,0.175118,0.147933,0.035407,0.097054,-0.021608,0.018880,15.99,0


In [9]:
## 샘플링 후 Class 비율

df_sample['Class'].value_counts(normalize=True)

,proportion
Class,
0,0.953107
1,0.046893


In [10]:
# Time 제거: 데이터셋 첫 거래로부터의 경과 초 → 예측 정보 없음 + 스케일이 커서 SMOTE 거리 계산 왜곡
df_sample = df_sample.drop(['Time'], axis=1)

In [11]:
# X, y로 데이터프레임 분리

X = df_sample.drop(['Class'], axis=1)
y = df_sample['Class']

## 3. 학습 데이터, 테스트 데이터 분할

- 8:2 비율로 분할
  - stratify = y 옵션으로 클래스 비율 유지
  - random_state = 42

In [12]:
# 분할, 클래스 비율 유지

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [13]:
# 분할된 데이터의 Class 비율 출력

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Class
0    0.953056
1    0.046944
Name: proportion, dtype: float64
Class
0    0.953311
1    0.046689
Name: proportion, dtype: float64


## 4. 데이터 전처리

- Amount 변수만 StandardScaler로 표준화 -> 새로운 변수 Amount_Sclaed로 대체
- Amount 원본 변수 제거
  - 이대로 X, y로 데이터프레임 분리

- 데이터 누수 방지
  - 전체 데이터로 StandardScaler 학습 시?
    - test의 mean, std가 train에 반영됨
  - train data에만

In [14]:
# StandardScaler 사용
## 정규화 -> Amount_Scaled로 대체 -> 원본 Amount 제거

scaler = StandardScaler()

X_train['Amount_Scaled'] = scaler.fit_transform(X_train[['Amount']])   # fit은 train만
X_test['Amount_Scaled']  = scaler.transform(X_test[['Amount']])        # test는 transform만

X_train = X_train.drop(['Amount'], axis=1)
X_test  = X_test.drop(['Amount'], axis=1)

## 5. SMOTE

- Class 비율이 정상: 대략 8000건, 사기: 대략 400건
  - 클래스 불균형이 심함
  - 모델의 학습, 추론 시 패턴 파악에 악영향 끼칠 수 있음

- 데이터 누수 방지 -> 학습 데이터에만 사용해야함
  - X_train, y_train에만 SMOTE를 사용해야 함

- 본 데이터셋에서 SMOTE 사용 시..?
  - SMOTE: 유클리디안 kNN으로 이웃 찾아 보간
  - 데이터의 스케일이 중요함 -> Time column
    - Time 이 다른 column 보다 scale이 큼 -> 거리 기반 이웃 탐색이 왜곡됨
    - Time drop 후 SMOTE 수행

In [15]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [16]:
# SMOTE 적용 후의 클래스 비율 확인

print("SMOTE 전:", y_train.value_counts().to_dict())
print("SMOTE 후:", y_train_smote.value_counts().to_dict())

SMOTE 전: {0: 7999, 1: 394}
SMOTE 후: {0: 7999, 1: 7999}


## 6. 적합한 모델 선택 후 학습

- metric
  - classification_report로 Precision, Recall, F1-score를 확인
  - average_precision_score로 PR-AUC를 계산하여 출력


- baseline
  - 튜닝되지 않은 로지스틱 회귀, rf, GDBT 모델 등등으로 baseline 설정
  - 이후 모델 튜닝, 스태킹, 앙상블 등등을 이용

In [17]:
## 1. 로지스틱 회귀

lr = LogisticRegression(random_state=42)
lr.fit(X_train_smote, y_train_smote)
y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_lr))
print(average_precision_score(y_test, y_proba_lr))

display(pd.DataFrame(classification_report(y_test, y_pred_lr, output_dict=True)).T.round(4))

              precision    recall  f1-score   support

           0       1.00      0.97      0.98      2001
           1       0.61      0.97      0.75        98

    accuracy                           0.97      2099
   macro avg       0.81      0.97      0.87      2099
weighted avg       0.98      0.97      0.97      2099

0.9557498175275924


,precision,recall,f1-score,support
0,0.9985,0.9700,0.9840,2001.00
1,0.6129,0.9694,0.7510,98.00
accuracy,0.9700,0.9700,0.9700,0.97
macro avg,0.8057,0.9697,0.8675,2099.00
weighted avg,0.9805,0.9700,0.9731,2099.00


In [18]:
## 2. 랜덤 포레스트

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_smote, y_train_smote)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print(average_precision_score(y_test, y_proba_rf))

display(pd.DataFrame(classification_report(y_test, y_pred_rf, output_dict=True)).T.round(4))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00      2001
           1       0.99      0.89      0.94        98

    accuracy                           0.99      2099
   macro avg       0.99      0.94      0.97      2099
weighted avg       0.99      0.99      0.99      2099

0.9479101113683667


,precision,recall,f1-score,support
0,0.9945,0.9995,0.9970,2001.0000
1,0.9886,0.8878,0.9355,98.0000
accuracy,0.9943,0.9943,0.9943,0.9943
macro avg,0.9916,0.9436,0.9662,2099.0000
weighted avg,0.9943,0.9943,0.9941,2099.0000


In [ ]:
# 랜덤 포레스트 모델 Class 0 PR-AUC 값 계산

print("Class 0 PR-AUC:", average_precision_score(1 - y_test, 1 - y_proba_rf))

In [19]:
## 3. Gradient Boosting

gb = GradientBoostingClassifier()
gb.fit(X_train_smote, y_train_smote)
y_pred_gb = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_gb))
print(average_precision_score(y_test, y_proba_gb))

display(pd.DataFrame(classification_report(y_test, y_pred_gb, output_dict=True)).T.round(4))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99      2001
           1       0.75      0.96      0.84        98

    accuracy                           0.98      2099
   macro avg       0.87      0.97      0.92      2099
weighted avg       0.99      0.98      0.98      2099

0.9585171411666575


,precision,recall,f1-score,support
0,0.9980,0.9840,0.9909,2001.0000
1,0.7460,0.9592,0.8393,98.0000
accuracy,0.9828,0.9828,0.9828,0.9828
macro avg,0.8720,0.9716,0.9151,2099.0000
weighted avg,0.9862,0.9828,0.9839,2099.0000


In [20]:
## 4. lgbm

lgbm = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
lgbm.fit(X_train_smote, y_train_smote)
y_pred_lgbm = lgbm.predict(X_test)
y_proba_lgbm = lgbm.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_lgbm))
print(average_precision_score(y_test, y_proba_lgbm))

display(pd.DataFrame(classification_report(y_test, y_pred_lgbm, output_dict=True)).T.round(4))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      2001
           1       0.90      0.89      0.89        98

    accuracy                           0.99      2099
   macro avg       0.95      0.94      0.94      2099
weighted avg       0.99      0.99      0.99      2099

0.9460550014700797


,precision,recall,f1-score,support
0,0.9945,0.9950,0.9948,2001.00
1,0.8969,0.8878,0.8923,98.00
accuracy,0.9900,0.9900,0.9900,0.99
macro avg,0.9457,0.9414,0.9435,2099.00
weighted avg,0.9899,0.9900,0.9900,2099.00


In [21]:
## 5. xgboost

xgb = XGBClassifier(random_state=42, n_jobs=-1, eval_metric="logloss")
xgb.fit(X_train_smote, y_train_smote)
y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_xgb))
print(average_precision_score(y_test, y_proba_xgb))

display(pd.DataFrame(classification_report(y_test, y_pred_xgb, output_dict=True)).T.round(4))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      2001
           1       0.89      0.89      0.89        98

    accuracy                           0.99      2099
   macro avg       0.94      0.94      0.94      2099
weighted avg       0.99      0.99      0.99      2099

0.9480875599360524


,precision,recall,f1-score,support
0,0.9945,0.9945,0.9945,2001.0000
1,0.8878,0.8878,0.8878,98.0000
accuracy,0.9895,0.9895,0.9895,0.9895
macro avg,0.9411,0.9411,0.9411,2099.0000
weighted avg,0.9895,0.9895,0.9895,2099.0000


## 6. 튜닝 및 앙상블, Threshold 조정

- 하이퍼파라미터 튜닝
  - random forest 모델 하이퍼파라미터 조정
  - RandomSearchCV 이용
  - SMOTE로 생성된 데이터 사용 시 합성 샘플이 validation fold로 들어오게 되어 CV 스코어가 지나치게 높아질 수 있음
    - 원본 X_train, y_train으로 fit

In [ ]:
from imblearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

pipe = Pipeline([("smote", SMOTE(random_state=42)),
                 ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))])

params = {"clf__n_estimators": [200, 400, 600],
          "clf__max_depth": [None, 10, 20],
          "clf__min_samples_leaf": [1, 2, 4],
          "clf__max_features": ["sqrt", 0.3]}

search = RandomizedSearchCV(pipe, params, n_iter=20, scoring="average_precision",
                            cv=StratifiedKFold(5, shuffle=True, random_state=42),
                            random_state=42, n_jobs=-1)
search.fit(X_train, y_train)          # ← SMOTE 안 걸린 원본 X_train
print(search.best_params_, search.best_score_)

{'clf__n_estimators': 400, 'clf__min_samples_leaf': 1, 'clf__max_features': 'sqrt', 'clf__max_depth': 20} 0.9239289602869276


- 앙상블: 스태킹 / 소프트 보팅
  - 베이스 모델이 서로 지나치게 닮음
    - RF / XGBoost, LGBM...
    - 앙상블을 함에 있어서 큰 의미 없어짐

  

  - Baseline random forest 에 비해서 개선 여지가 물리적으로 없음

---

- Threshold 조정

In [22]:
from sklearn.metrics import precision_recall_curve, f1_score

p, r, t = precision_recall_curve(y_test, y_proba_rf)
f1 = 2*p*r / (p+r+1e-12)
i = f1[:-1].argmax()

print(f"최적 threshold={t[i]:.3f}, F1={f1[i]:.4f} / 기본 0.5 F1={f1_score(y_test, y_pred_rf):.4f}")

최적 threshold=0.520, F1=0.9355 / 기본 0.5 F1=0.9355


In [24]:
# test 데이터로 튜닝된 모델 성능 평가

best = RandomForestClassifier(n_estimators=400, max_depth=20, min_samples_leaf=1,
                              max_features="sqrt", random_state=42, n_jobs=-1)
best.fit(X_train_smote, y_train_smote)

y_pred_best  = best.predict(X_test)
y_proba_best = best.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred_best))
print("PR-AUC:", average_precision_score(y_test, y_proba_best))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00      2001
           1       0.99      0.88      0.93        98

    accuracy                           0.99      2099
   macro avg       0.99      0.94      0.96      2099
weighted avg       0.99      0.99      0.99      2099

PR-AUC: 0.950055353484894
